# End-to-End Movie Pipeline: Optimisation → Predictions → Frames

This notebook runs the full pipeline for one source→target pair:

1. **Optimise** – run Ledidi, save every accepted sequence to disk as `seq_N.pt`
2. **Predict** – run Akita-PyTorch on each saved sequence to get a Hi-C contact map
3. **Scan** – run FIMO to find CTCF motifs in each sequence
4. **Plot** – save one combined PNG frame per sequence
5. **Assemble** – call ffmpeg to turn the frames into an mp4

Example pair used here:
- **Source:** `chr3:138672128–139982848`
- **Target:** `chr5:43542528–44853248`

In [1]:
import os
import re
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from memelite import fimo

# pytorch_akita appended (not inserted) so its utils/ does not shadow ours
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))
sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi/"))
sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))

from ledidi import ledidi

from utils.model_utils import load_model
from utils.fimo_utils import read_meme_pwm, run_fimo, ctcf_hits_from_fimo
from utils.data_utils import from_upper_triu
from utils.plot_utils import save_movie_frame

## Parameters

In [2]:
# ── Model & motif ─────────────────────────────────────────────────────────────
MODEL_PATH    = ("/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth")
CTCF_PWM_PATH = "/home1/smaruj/ledidi_akita/data/pwm/MA0139.1.meme"

# ── Source / target sequence pair ─────────────────────────────────────────────
BASE_DIR    = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/optimizations/genome_rewiring"
FOLD        = 0

SRC_CHROM, SRC_START, SRC_END     = "chr3", 138672128, 139982848
TGT_CHROM, TGT_START, TGT_END     = "chr5",  43542528,  44853248

X_PATH     = f"{BASE_DIR}/ohe_X_fold{FOLD}/{SRC_CHROM}_{SRC_START}_{SRC_END}_X.pt"
Y_BAR_PATH = f"{BASE_DIR}/genomic_targets_fold{FOLD}/{TGT_CHROM}_{TGT_START}_{TGT_END}_target.pt"

# ── Output directories ────────────────────────────────────────────────────────
RUN_TAG    = f"{SRC_CHROM}_{SRC_START}_{SRC_END}_to_{TGT_CHROM}_{TGT_START}_{TGT_END}"
SEQ_DIR    = f"{BASE_DIR}/movie_seqs/"   # accepted sequences (.pt)
FRAMES_DIR = f"{BASE_DIR}/movie_frames/" # PNG frames
MOVIE_PATH = f"{BASE_DIR}/movies/{RUN_TAG}.mp4"

# ── Ledidi hyperparameters ────────────────────────────────────────────────────
LEDIDI_L                   = 0.05
LEDIDI_MAX_ITER            = 2000
LEDIDI_EARLY_STOPPING_ITER = 2000

# ── Visualisation ─────────────────────────────────────────────────────────────
VMIN          = -0.6   # contact-map colour scale
VMAX          =  0.6
MAX_CTCF_HITS =  3     # y-axis limit for CTCF bar track
MAP_SIZE      = 512    # Akita output is 512×512
CROP_OFFSET   = 64     # Akita crops 64 bins on each side of the 640-bin input
BIN_SIZE      = 2048   # bp per bin

# ── FIMO ──────────────────────────────────────────────────────────────────────
FIMO_THRESHOLD = 1e-4

# ── ffmpeg ────────────────────────────────────────────────────────────────────
MOVIE_FPS = 10

for d in [SEQ_DIR, FRAMES_DIR, os.path.dirname(MOVIE_PATH)]:
    os.makedirs(d, exist_ok=True)

print(f"Source : {SRC_CHROM}:{SRC_START}-{SRC_END}")
print(f"Target : {TGT_CHROM}:{TGT_START}-{TGT_END}")
print(f"Sequences → {SEQ_DIR}")
print(f"Frames    → {FRAMES_DIR}")
print(f"Movie     → {MOVIE_PATH}")

Source : chr3:138672128-139982848
Target : chr5:43542528-44853248
Sequences → /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/optimizations/genome_rewiring/movie_seqs/
Frames    → /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/optimizations/genome_rewiring/movie_frames/
Movie     → /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/optimizations/genome_rewiring/movies/chr3_138672128_139982848_to_chr5_43542528_44853248.mp4


## Load Model and CTCF PWM

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = load_model(MODEL_PATH, device)

pwm_CTCF    = read_meme_pwm(CTCF_PWM_PATH)
motifs_dict = {"CTCF": pwm_CTCF}

Using device: cuda:0


## Step 1 – Optimisation

Run Ledidi with `return_history=True`. `history['edits']` is a list of
`(1, 4, L)` tensors — one per **accepted** step, in chronological order.
These are saved to `SEQ_DIR` as `seq_0.pt`, `seq_1.pt`, …

In [4]:
X     = torch.load(X_PATH,     weights_only=True, map_location=device)
y_bar = torch.load(Y_BAR_PATH, weights_only=True, map_location=device)

print(f"Running Ledidi: {SRC_CHROM}:{SRC_START}-{SRC_END}  →  {TGT_CHROM}:{TGT_START}-{TGT_END}")

_ = ledidi(
    model, X, y_bar,
    batch_size          = 1,
    l                   = LEDIDI_L,
    max_iter            = LEDIDI_MAX_ITER,
    early_stopping_iter = LEDIDI_EARLY_STOPPING_ITER,
    input_loss          = torch.nn.L1Loss(reduction="sum"),
    output_loss         = torch.nn.L1Loss(reduction="sum"),
    return_history      = False,
    verbose             = True,
    device              = device,
)

# Free GPU memory
del X, y_bar
torch.cuda.empty_cache()

Running Ledidi: chr3:138672128-139982848  →  chr5:43542528-44853248


/home1/smaruj/miniconda3/envs/pytorch_hic/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:84.)
  return F.conv1d(input, weight, bias, self.stride,


iter=I	input_loss=0.0	output_loss=4.009e+04	total_loss=4.009e+04	time=0.0
iter=100	input_loss=1.449e+05	output_loss=5.802e+03	total_loss=1.304e+04	time=13.15
iter=200	input_loss=1.288e+05	output_loss=4.386e+03	total_loss=1.083e+04	time=11.65
iter=300	input_loss=1.464e+05	output_loss=3.986e+03	total_loss=1.131e+04	time=11.33
iter=400	input_loss=1.647e+05	output_loss=4.461e+03	total_loss=1.27e+04	time=11.33
iter=500	input_loss=1.857e+05	output_loss=4.934e+03	total_loss=1.422e+04	time=11.33
iter=600	input_loss=2.019e+05	output_loss=4.43e+03	total_loss=1.452e+04	time=11.34
iter=700	input_loss=2.165e+05	output_loss=3.725e+03	total_loss=1.455e+04	time=11.34
iter=800	input_loss=2.231e+05	output_loss=3.719e+03	total_loss=1.487e+04	time=11.34
iter=900	input_loss=2.291e+05	output_loss=5.133e+03	total_loss=1.659e+04	time=11.35
iter=1000	input_loss=2.332e+05	output_loss=4.858e+03	total_loss=1.652e+04	time=11.35
iter=1100	input_loss=2.392e+05	output_loss=4.475e+03	total_loss=1.644e+04	time=11.35
it

## Steps 2–4 – Predict Contact Maps, Scan CTCF, Save Frames

Loads each saved `seq_N.pt`, runs Akita, runs FIMO, and saves a PNG frame.
GPU memory is cleared after each sequence to avoid OOM errors.

In [8]:
# Discover and sort all saved sequences
seq_files = sorted(
    [f for f in os.listdir(SEQ_DIR) if re.match(r"seq_\d+\.pt$", f)],
    key=lambda f: int(re.search(r"\d+", f).group()),
)
print(f"Found {len(seq_files)} sequences to process")

model.eval()
for frame_idx, fname in enumerate(seq_files):
    seq = torch.load(os.path.join(SEQ_DIR, fname), weights_only=True)  # (1, 4, L) CPU
    
    # Akita prediction
    seq_gpu = seq.to(device)
    with torch.no_grad():
        pred = model(seq_gpu).cpu()  # (1, 1, 130305)
    contact_map = from_upper_triu(pred[0], matrix_len=512, num_diags=2)
    
    hits   = run_fimo(seq, motifs_dict, FIMO_THRESHOLD)
    
    ctcf_plus, ctcf_minus = ctcf_hits_from_fimo(hits)

    # Crop CTCF tracks to match the 512-bin Akita cropped window (bins 64–575)
    ctcf_plus  = ctcf_plus[CROP_OFFSET : CROP_OFFSET + MAP_SIZE]
    ctcf_minus = ctcf_minus[CROP_OFFSET : CROP_OFFSET + MAP_SIZE]
    
    # Save frame
    save_movie_frame(contact_map, ctcf_plus, ctcf_minus, frame_idx, FRAMES_DIR)

    del seq_gpu, pred, seq
    torch.cuda.empty_cache()

    if (frame_idx + 1) % 10 == 0 or frame_idx == len(seq_files) - 1:
        print(f"  {frame_idx + 1}/{len(seq_files)} frames saved")

print(f"\nDone — {len(seq_files)} frames saved to {FRAMES_DIR}")

Found 45 sequences to process
  10/45 frames saved
  20/45 frames saved
  30/45 frames saved
  40/45 frames saved
  45/45 frames saved

Done — 45 frames saved to /project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita/optimizations/genome_rewiring/movie_frames/


## Steps 5 – Create the movie

ffmpeg -framerate 3 -i movie_frames/frame_%03d.png \
       -c:v libx264 -pix_fmt yuv420p \
       -movflags +faststart movie.mp4
